# 01_data_preprocess

Builds the per-frame **manifest** (frame -> video_id -> identity_id) that the leakage-safe
group split depends on. Logic lives in `src/data_prep.py`; this notebook just drives it.

DeepfakeBench preprocessing runs **per dataset** as access clears. EULA-gated sets
(FF++, Celeb-DF, DeeperForensics, WildDeepfake) need their form approved first; public sets
(Deepfake-Eval-2024, DiffusionFace, DiFF, DFDC-Preview) can be processed now.

In [ ]:
# CELL 1 - bootstrap (restore creds, cd to repo, put src/ on path)
import os, sys, shutil, subprocess
DRIVE_ROOT = "/content/drive/MyDrive/CDTS_Research"
REPO = f"{DRIVE_ROOT}/deepfake-trust-research"
from google.colab import drive; drive.mount('/content/drive')
for f in ['.gitconfig', '.git-credentials']:
    s = f"{DRIVE_ROOT}/{f}"
    if os.path.exists(s):
        shutil.copy(s, f"/root/{f}")
        if f == '.git-credentials': os.chmod(f"/root/{f}", 0o600)
subprocess.run('git config --global credential.helper "store --file /root/.git-credentials"', shell=True)
os.chdir(REPO); sys.path.insert(0, os.path.join(REPO, "src"))
def sh(c):
    p = subprocess.run(c, shell=True, capture_output=True, text=True)
    print(p.stdout.strip())
    if p.stderr.strip(): print(p.stderr.strip())
print("bootstrap OK:", REPO)

In [ ]:
# CELL 2 - config + access/status table (runnable now)
!pip -q install pyyaml pandas
import importlib, data_prep; importlib.reload(data_prep)
cfg = data_prep.load_configs(REPO)
print("enabled datasets:", list(data_prep.enabled_datasets(cfg["datasets"])))
data_prep.status_table(REPO)   # 'processed' column shows which DeepfakeBench has rearranged

### Run DeepfakeBench preprocessing (per dataset, once its raw data is placed)
DeepfakeBench is config-driven: place the raw dataset, point `external/DeepfakeBench/preprocessing/config.yaml` at it (per their README), then run the cell below. It does face detect/align/crop then writes the dataset JSON. Start with a public set while EULA forms clear.

In [ ]:
# CELL 3 - DeepfakeBench preprocess + rearrange (uncomment when raw data is placed)
# rc1, rc2 = data_prep.run_dfb_preprocess(cfg["paths"]["deepfakebench"])
# print("preprocess rc:", rc1, " rearrange rc:", rc2)

In [ ]:
# CELL 4 - build manifests for whatever is already processed
import pandas as pd
era = data_prep.era_lookup(cfg["timeline"])
dfb = cfg["paths"]["deepfakebench"]; frames_root = cfg["paths"]["frames"]
out = cfg["paths"]["results"]; os.makedirs(out, exist_ok=True)
mans = []
for name in data_prep.enabled_datasets(cfg["datasets"]):
    jp = data_prep.dfb_json_path(dfb, name)
    if not jp:
        print(f"  pending: {name}"); continue
    m = data_prep.build_manifest_from_json(name, jp, era_map=era, frames_root=frames_root)
    m.to_csv(os.path.join(out, f"manifest_{name}.csv"), index=False); mans.append(m)
    print(f"  {name}: {len(m)} frames | {m.identity_id.nunique()} identities | fake-rate {m.label.mean():.2f}")
if mans:
    full = pd.concat(mans, ignore_index=True)
    full.to_csv(os.path.join(out, "manifest_all.csv"), index=False)
    print("combined:", len(full), "frames ->", os.path.join(out, "manifest_all.csv"))
else:
    print("nothing processed yet - manifests build automatically once a dataset JSON exists")

In [ ]:
# CELL 5 - commit (end-of-unit ritual)
sh(f'cd "{REPO}" && git add -A && git commit -m "01_data_preprocess: data_prep module + frame manifests" && git push')